In [1]:
import textwrap

def visualise_texts(texts: list[str], width: int = 100):
    for text in texts:
        print('\n'.join(textwrap.wrap(text, width, break_long_words=False)))
        print('=' * width)
        print('=' * width)

def write_md(filename: str, text: str):
    with open(filename, "w") as f:
        f.write(text)

## Visualise SourceDocuments

In [2]:
from ddd_policy_tracer.discovery.adapters import FilesystemSourceDocumentRepository

doc_repo = FilesystemSourceDocumentRepository("data/committee_documents.jsonl")
doc = doc_repo.get_latest(source_id="committees", source_document_id='https://www.aph.gov.au/DocumentStore.ashx?id=37e069c7-5c5b-4dc1-9b19-074d66dbe9f1&subId=778129')

## Visualise Chunks

In [5]:
from ddd_policy_tracer.analysis.chunks.adapters import SQLiteDocumentChunkRepository
from ddd_policy_tracer.utils.db_manager import DbManager
import pandas as pd

repository = SQLiteDocumentChunkRepository(db_manager=DbManager.get_instance())
chunks = await repository.list_chunks(source_id="committees")
df = pd.DataFrame([c.__dict__ for c in chunks])

In [6]:
docs = doc_repo.list_versions(source_id="committees")

In [ ]:
for chunk in chunks:
    doc = doc_repo.get_latest(source_document_id=chunk.source_document_id)
    chunk.metadata["doc"]

In [ ]:
visualise_texts(df.sample(5).chunk_text)

In [3]:
from pathlib import Path

from ddd_policy_tracer.utils.pdf.pdf_parser import PdfParser

parser = PdfParser(min_font_size=8)
pdf_bytes = Path("/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/FINIGHAN-NEELAM-Reframing-the-AustraliaJapan-energy-relationship-2.0.pdf").read_bytes()
md_text = parser.parse(pdf_bytes)

In [1]:
import httpx
from bs4 import BeautifulSoup
from ddd_policy_tracer.discovery.source_strategies import LowyInstituteSourceStrategy
from ddd_policy_tracer.utils.pdf.pdf_parser import PdfParser
import pymupdf4llm
import pymupdf

parser = PdfParser()
lowy = LowyInstituteSourceStrategy()
# res = httpx.get("https://www.lowyinstitute.org/publications/australia-s-strategic-priorities-challenges-southeast-asia")
# soup = BeautifulSoup(res.text, "html.parser")
# download_link = soup.find("a", {"class": "download-link", "href": True})

In [16]:
result = lowy.extract_document(source_url="https://www.lowyinstitute.org/publications/reframing-australia-japan-energy-relationship")

In [17]:
with open("pymupdf.md", "w") as f:
    f.write(result.extracted_text)

In [4]:
import pdf_inspector

from pathlib import Path

from ddd_policy_tracer.utils.pdf.analyse_fonts import detect_paragraph_font_size


paths = [
    Path("Sub02.pdf"),
    Path("Sub01.pdf"),
    Path("Sub17.pdf"),
]

def write_md(filename: str, text: str):
    with open(filename, "w") as f:
        f.write(text)

for path in paths:
    pdf_bytes = path.read_bytes()
    pdf_inspect_result = pdf_inspector.process_pdf_bytes(pdf_bytes)
    if not pdf_inspect_result.markdown:
        print(f"Failed to process {path}")
        continue
    filename = str(path).split(".")[0]
    write_md("pdf_inspector_" + filename + ".md", pdf_inspect_result.markdown)

    # custom parser
    doc = pymupdf.Document(stream=pdf_bytes)
    p_font_size = detect_paragraph_font_size(doc)
    parser = PdfParser(min_font_size=p_font_size)
    text = parser.parse(pdf_bytes)
    write_md("pymupdf_paragraphs_" + filename + ".md", text)

In [2]:
from bs4 import BeautifulSoup
import httpx

BASE_URL = "https://www.aph.gov.au"
url = (
    BASE_URL
    + "/Parliamentary_Business/Committees/Senate/Environment_and_Communications/CRA48P/Submissions"
)
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:143.0) Gecko/20100101 Firefox/141.0"
}

res = httpx.get(url, headers=headers)

In [3]:
soup = BeautifulSoup(res.text, "html.parser")
table = soup.find("table", {"class": "rgMasterTable"})
if not table:
    raise ValueError("Could not find table with class 'rgMasterTable'")
link_tags = []
for row in table.find_all("tr", class_=["rgRow", "rgAltRow"]):
    first_anchor = row.find("a", {"class": "fileLink", "href": True})
    link_tags.append(first_anchor) if first_anchor else None
pdf_links = [BASE_URL + tag["href"] for tag in link_tags]

In [2]:
from ddd_policy_tracer.utils.pdf.pdf_parser import PdfParser
from pathlib import Path

def write_md(filename: str, text: str):
    with open(filename, "w") as f:
        f.write(text)

pdf_bytes = Path("committee_inquiry.pdf").read_bytes()
parser = PdfParser()
md_text = parser.parse(pdf_bytes)
write_md("committee_inquiry.md", md_text)

In [5]:
from ddd_policy_tracer.paths import DATA_DIR

len([str(f) for f in (DATA_DIR / "committee_artifacts").iterdir()])

60

In [6]:
import asyncio

from graphrag_sdk import ConnectionConfig, GraphRAG

from ddd_policy_tracer.graph_rag.embedder import LocalEmbedder, LocalLLM
from ddd_policy_tracer.paths import DATA_DIR
import logging

rag = GraphRAG(
        connection=ConnectionConfig(
            host="localhost",
            graph_name="my_graph",
        ),  # graph_name = per-tenant isolation
        llm=LocalLLM(model_name="hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF:latest"),
        embedder=LocalEmbedder(),
        embedding_dimension=1024,
    )

/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7001.23it/s]


In [14]:
answer = await rag.retrieve("What are the most important recommendations to the committee?")

In [16]:
answer.items

[RetrieverResultItem(content='## Key Entities\n- ACIL Committee Members: Includes Tyrone Shandiman, Bradley von Xanten, Eva Jones, John Devaney, Carol Roberts, Peter Marer, and general committee members.\n- Carol Roberts: Member of the committee supporting consumer interests.\n- Senate Environment and Communications References Committee\n- Climate Risk Assessment\n- ASLCG\n- ONI\n- Senate Environment & Communications References Committee\n- Lawrence et al.\n- Australian Institute for Earth System Science\n- Environment and Communications References Committee\n- Australian Red Cross\n- National Climate Risk Assessment\n- Senate\n- Department of Climate Change, Energy, the Environment and Water\n- Select Committee\n- TCC\n- HIA\n- Select Committee on Climate Risk\n- ACIL', metadata={'section': 'entities'}, score=None),
 RetrieverResultItem(content='## Entity Relationships\n- ACIL Committee Members —[MEMBER]→ ACIL: (ACIL Committee Members, MEMBER, ACIL): Committee members are part of the 

In [12]:
import textwrap

print(textwrap.fill(answer.answer, width=80))

Based on the content of the HIA Submission 5 and the overall focus of the
document, the most important recommendations to the Senate committee would
likely include:  1. **Increased Transparency**: The committee should be provided
with the full Climate Risk Assessment, as withholding it from the public has
been a significant concern. Greater transparency is essential for informed
decision-making and public trust.  2. **Budgetary and Implementation Planning**:
Detailed information on the costs of climate-driven natural disasters and the
costs associated with government adaptation plans is crucial for proper
budgeting and strategic planning.  3. **Enhanced Building Codes and Standards**:
The committee should be informed about the current building codes and how they
can be updated to better withstand climate-related risks. This includes
preparing for more stringent standards in new construction and retrofitting
existing buildings.  4. **Regular Updates and Monitoring**: The government
shou

In [13]:
answer


RagResult(answer="Based on the content of the HIA Submission 5 and the overall focus of the document, the most important recommendations to the Senate committee would likely include:\n\n1. **Increased Transparency**: The committee should be provided with the full Climate Risk Assessment, as withholding it from the public has been a significant concern. Greater transparency is essential for informed decision-making and public trust.\n\n2. **Budgetary and Implementation Planning**: Detailed information on the costs of climate-driven natural disasters and the costs associated with government adaptation plans is crucial for proper budgeting and strategic planning.\n\n3. **Enhanced Building Codes and Standards**: The committee should be informed about the current building codes and how they can be updated to better withstand climate-related risks. This includes preparing for more stringent standards in new construction and retrofitting existing buildings.\n\n4. **Regular Updates and Monitor

In [1]:
from ddd_policy_tracer.analysis.chunks.adapters import SQLiteDocumentChunkRepository
from ddd_policy_tracer.utils.db_manager import DbManager
import pandas as pd

repository = SQLiteDocumentChunkRepository(db_manager=DbManager.get_instance())
chunks = await repository.list_chunks(source_id="hansard")
df = pd.DataFrame([c.__dict__ for c in chunks])

/Users/thomasscelsi/Documents/programming/ddd-policy-tracer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
mask = df.chunk_text.str.lower().str.contains("israel")
pd.json_normalize(df[mask].metadata.apply(lambda x: x.get("doc_metadata").get("talker")))

AttributeError: 'NoneType' object has no attribute 'get'

In [2]:
df.sample(5)

,chunk_id,source_id,source_document_id,document_checksum,chunk_index,start_char,end_char,chunk_text,topics,embedding,metadata
5287,chunk_b7e42a716dffa015,hansard,256063_2025-11-24_senate_main_23_0_none_2_0,62343b063d8c0126fd50ca2d99d1425b6568330ad0bd32...,0,0,1123,I don't think the Australian people are silly....,"[{'topic': 'Population and Demographics', 'con...","[-0.03857421875, -0.05419921875, -0.0112915039...",None
8013,chunk_17b4226e78438664,hansard,282997_2025-11-26_senate_main_23_0_none_2_0,7cbc885f501d4d5cf4740fc763987b4dffa65d8987be1a...,0,0,1334,What extraordinary contributions we have heard...,"[{'topic': 'Science and Innovation', 'confiden...","[-0.06689453125, -0.04345703125, -0.0075073242...",None
5595,chunk_cd4b9534fca2c562,hansard,259819_2025-10-28_senate_main_2_0_0_23_0,46b5754a2f8626551330d549eb9dde9e5a8de851599273...,0,0,726,That's a matter for the minister. I have said ...,[{'topic': 'Digital Infrastructure and Telecom...,"[-0.061767578125, -0.0021820068359375, -0.0094...",None
3286,chunk_cd8bc01925c52c88,hansard,217241_2025-02-13_senate_main_26_0_none_5_4,dbd2ee3078c178ed8d16b91d4ed8c90977734b2d4f03a7...,0,0,1273,"It is ancient history, but the Liberal Nationa...","[{'topic': 'Electoral and Democratic Reform', ...","[-0.003814697265625, -0.06689453125, -0.009521...",None
11302,chunk_67dc75a78ab33734,hansard,316052_2025-10-09_hor_main_4_0_0_4_0,5839c74cda6c483c0ba921a371ee319deb97563ca20168...,0,0,1340,I rise to speak on the Australian Centre for D...,[{'topic': 'Public Health and Disease Preventi...,"[-0.0262451171875, 0.00927734375, -0.003097534...",None
